# 10.5 结果、日志与故障定位

## 本节目标

- 读取历史 CSV 的应用与 solver 字段
- 区分 assembly、linear solver 和 total
- 按部署阶段定位故障

## 环境检查

检查 Ascend NPU/CANN 环境、运行日志和 msprof 输出；初始化失败不能作为性能样本。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("说明：本章默认运行 Xyce Adapter benchmark，不宣称完整 upstream Xyce 仿真。")


## 历史结果

原 README 记录（历史 Host prototype 数据，非当前 NPU 实测）Host CPU/OpenMP/CANN 9.0.0 下六矩阵的 CPU single、OpenMP16 和 optimized solver。总模拟加速相对 CPU single 为 1.87x–4.42x，linear solver 加速为 2.29x–5.27x；所有矩阵收敛，solution error 约 `7e-4`–`2.2e-3`。

`total_simulation_ms` 包含 assembly/adapter/prepare/solve；评估 backend 看 `linear_solver_ms` 与 profiling，评估 wrapper 应用看 total。

## 故障分层

1. dependency：`prepare_backend.sh` 找不到 CMakeLists 或版本不匹配
2. build：编译器/CMake/OpenMP 或头文件失败
3. input：CSR 缓存损坏或矩阵生成失败
4. solve：不收敛、残差/误差超限
5. result：CSV 路径不可写、字段为空或 solver 行数不对

当前脚本没有独立 log 文件；stdout/stderr 是运行日志，需要留存时可用 `2>&1 | tee results/run.log`。自动生成的日志不提交课程仓库。

## 课后实践

设计一份最小故障报告，包含环境、命令、退出码、最后日志、CSV 和正确性字段。参考答案见 `answer/10.05_answer.md`。

## 实验记录与练习

日志必须同时包含真实 backend、Device、迭代数、残差和当前 CSV；历史 HostPrototype 数据只作历史对照。

完成后回答：实际后端是什么？reference 与 tolerance 是什么？主要耗时来自计算、通信、传输还是同步？改变一个并发或算法参数后，正确性和性能如何变化？参考答案仅通过本章 `answer/` 链接查阅。
